# Notebook 02: Configuration Deep Dive

**Module:** 12 Capstone — water-bodies-detection  
**Duration:** ~2.5 hrs

---

## Learning Objectives

1. Walk through every section of config/default.yaml
2. Explain why each default value was chosen
3. Know which params affect train vs infer vs post
4. Practice modifying config safely

## Single Source of Truth

**File:** `water-bodies-detection/config/default.yaml`

All scripts load this YAML — never hardcode hyperparameters in Python.

In [ ]:
import os, json, yaml
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 5)
REPO = '../../water-bodies-detection'
print('Capstone repo path:', os.path.abspath(REPO) if os.path.exists(REPO) else 'clone water-bodies-detection alongside Machine-Learning')

In [ ]:
cfg_path = os.path.join(REPO, 'config/default.yaml')
if os.path.isfile(cfg_path):
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    for section in ['data', 'tiling', 'model', 'training', 'prediction', 'postprocess']:
        print(f'\n=== {section} ===')
        print(yaml.dump({section: cfg.get(section, {})}, default_flow_style=False)[:600])
else:
    print('Open config/default.yaml in the repo')

## Section Guide

### `data`
| Key | Default | Why |
|-----|---------|-----|
| `input_size` | 512 | Matches tile size, GPU memory |
| `band_indices` | [2,3,4,6,7,8] | Blue→NIR multispectral (not RGB) |
| `batch_size` | 2 | 512×512×6ch fits ~8GB VRAM |
| `validation_split` | 0.15 | Tile-level holdout |

### `tiling`
| Key | Default | Why |
|-----|---------|-----|
| `overlap_fraction` | 0.5 | Training coverage + inference blending |
| `boundary_width_meters` | 1.0 | Bund dilation width in meters |
| `negative_tile_ratio` | 0.2 | Reduce false positives on bare soil |
| `percentile_low/high` | 2/98 | Robust normalization across scenes |

### `training`
| Key | Default | Why |
|-----|---------|-----|
| `stage1_epochs` | 12 | Decoder-only with frozen encoder |
| `learning_rate_stage1` | 2e-4 | Higher LR for random decoder |
| `learning_rate_stage2` | 2e-5 | 10× lower when unfreezing encoder |
| `loss_weight_boundary` | 0.35 | Sparse boundary — don't dominate gradients |

### `prediction` vs `postprocess` thresholds
| Stage | `threshold_aqua` | Purpose |
|-------|-----------------|----------|
| Inference | 0.5 | Recall-friendly probability maps |
| Post-process | 0.8 | Precision-friendly GIS polygons |

---

## Summary

default.yaml drives every stage — understand before changing any value.

**Next:** [03_Tiling_Pipeline.ipynb](03_Tiling_Pipeline.ipynb)